In [6]:
import ee
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

## Datasets used:
- Fire: https://developers.google.com/earth-engine/datasets/catalog/NASA_LANCE_SNPP_VIIRS_C2
- Weather: https://developers.google.com/earth-engine/datasets/catalog/IDAHO_EPSCOR_GRIDMET
- Vegetation: https://developers.google.com/earth-engine/datasets/catalog/JAXA_GCOM-C_L3_LAND_LAI_V3

In [3]:
ee.Authenticate()
ee.Initialize(project="cs578-wildfire-prediction")

In [8]:
# Define bounding box and dates
bbox = ee.Geometry.Polygon([
    [[-119.08580653031808, 34.710191016920575],
     [-119.08581587345162, 34.01019101696305],
     [-117.58651951593266, 34.01019101696305],
     [-117.5865288590662, 34.710191016920575]]
])

start_date = datetime.date(2024, 12, 1)
end_date = datetime.date(2024, 12, 31)
time_delta = end_date - start_date

In [ ]:
# for i in range(time_delta.days + 1):
#     day = start_date + datetime.timedelta(days=i)
#     next_day = day + datetime.timedelta(days=1)
    
#     day = str(day)
#     next_day = str(next_day)
    
#     # https://developers.google.com/earth-engine/datasets/catalog/NASA_LANCE_SNPP_VIIRS_C2
#     viirs_fire = ee.ImageCollection('NASA/LANCE/SNPP_VIIRS/C2') \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .filter(ee.Filter.neq('confidence', 0)) \
#         .select('frp') \
#         .map(lambda img: img.toFloat()) \
#         .first()


#     # https://developers.google.com/earth-engine/datasets/catalog/IDAHO_EPSCOR_GRIDMET
#     weather = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET") \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .map(lambda img: img.toFloat()) \
#         .first()

#     # https://developers.google.com/earth-engine/datasets/catalog/JAXA_GCOM-C_L3_LAND_LAI_V3
#     vegetation = ee.ImageCollection("JAXA/GCOM-C/L3/LAND/LAI/V3") \
#         .filterDate(day, next_day) \
#         .filterBounds(bbox) \
#         .select('LAI_AVE') \
#         .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) \
#         .map(lambda img: img.toFloat()) \
#         .first()
    
#     combined = viirs_fire.addBands(weather).addBands(vegetation)

#     # Export the entire collection as a stacked GeoTIFF
#     export_params = {
#         'image': combined,
#         'description': f"Fire prediction data for {str(day)}",
#         'folder': 'Fire_Pred',
#         'fileNamePrefix': str(day).replace("-", ""),
#         'region': bbox,
#         'scale': 375,  # 375m resolution
#         'crs': 'EPSG:4326',
#         'fileFormat': 'GeoTIFF',
#         'maxPixels': 1e9
#     }

#     # Start the export (check Tasks tab in GEE Console)
#     task = ee.batch.Export.image.toDrive(**export_params)
#     task.start()
#     print(day)

2024-12-01
2024-12-02
2024-12-03
2024-12-04
2024-12-05
2024-12-06
2024-12-07
2024-12-08
2024-12-09
2024-12-10
2024-12-11
2024-12-12
2024-12-13
2024-12-14
2024-12-15
2024-12-16
2024-12-17
2024-12-18
2024-12-19
2024-12-20
2024-12-21
2024-12-22
2024-12-23
2024-12-24
2024-12-25
2024-12-26
2024-12-27
2024-12-28
2024-12-29
2024-12-30
2024-12-31


In [14]:
var_info = {
    "frp": {
        "long_name": "Fire Radiative Power",
        "units": "MW"
    },
    "pr": {
        "long_name": "Precipitation",
        "units": "mm, daily total"
    },
    "rmax": {
        "long_name": "Maximum relative humidity",
        "units": "%"
    },
    "rmin": {
        "long_name": "Minimum relative humidity",
        "units": "%"
    },
    "sph": {
        "long_name": "Specific humididy",
        "units": "Mass fraction"
    },
    "srad": {
        "long_name": "Surface downward shortwave radiation",
        "units": "W/m^2"
    },
    "th": {
        "long_name": "Wind direction",
        "units": "deg"
    },
    "tmmn": {
        "long_name": "Minimum temperature",
        "units": "K"
    },
    "tmmx": {
        "long_name": "Maximum temperature",
        "units": "K"
    },
    "vs": {
        "long_name": "Wind velocity at 10m",
        "units": "m/s"
    },
    "erc": {
        "long_name": "Energy release component",
        "units": "NFDRS fire danger index"
    },
    "eto": {
        "long_name": "Daily grass reference evapotranspiration",
        "units": "mm"
    },
    "bi": {
        "long_name": "Burning index",
        "units": "NFDRS fire danger index"
    },
    "fm100": {
        "long_name": "100-hour dead fuel moisture",
        "units": "%"
    },
    "fm1000": {
        "long_name": "1000-hour dead fuel moisture",
        "units": "%"
    },
    "etr": {
        "long_name": "Daily alfalfa reference evapotranspiration",
        "units": "mm"
    },
    "vpd": {
        "long_name": "Mean vapor pressure deficit",
        "units": "kPa"
    },
    "lai_ave": {
        "long_name": "The sum of the one-sided green leaf area per unit ground area.",
        "units": "One-sided green leaf area per unit ground area"
    }
}

In [15]:
import datetime
import glob
import os

def merge_to_netcdf_with_separate_vars(tif_folder, output_nc):
    """Merge daily GeoTIFFs into NetCDF with separate variables"""
    # Get all GeoTIFF files
    tif_files = sorted(glob.glob(os.path.join(tif_folder, '*.tif')))
    
    # Initialize lists to store data arrays
    variables = {}
    
    # Process each file
    for f in tif_files:
        try:
            # Extract date from filename
            date_str = os.path.basename(f).split('.')[0]
            date = pd.to_datetime(date_str)
            
            # Open the GeoTIFF
            ds = rioxarray.open_rasterio(f)
            
            # Get band names (assuming they're preserved in the GeoTIFF)
            if not variables:  # First file - initialize variables
                num_bands = len(ds.band)
                # Create default band names if not available
                band_names = ds.attrs.get('long_name', [f'band_{i}' for i in range(num_bands)])
                if isinstance(band_names, str):  # Handle case where it's a string
                    band_names = eval(band_names)  # Convert string to list
                
                for i, band_name in enumerate(band_names):
                    band_name = band_name.lower()
                    variables[band_name] = {
                        'data': [],
                        'attrs': {
                            'long_name': var_info[band_name]["long_name"],  # Set long_name to just the band name
                            'units': var_info[band_name]["units"]
                        }
                    }
            
            # Append data for each band
            for i, band_name in enumerate(variables.keys()):
                band_name = band_name.lower()
                band_data = ds.isel(band=i)
                variables[band_name]['data'].append(band_data.expand_dims(time=[date]))
                
        except Exception as e:
            print(f"Error processing {f}: {str(e)}")
            continue
    
    if not variables:
        raise ValueError("No valid data found in any GeoTIFF")
    
    # Create final dataset
    ds_out = xr.Dataset()
    for var_name, var_dict in variables.items():
        var_data = xr.concat(var_dict['data'], dim='time')
        var_data = var_data.rename({'x': 'lon', 'y': 'lat'})
        var_data = var_data.drop_vars(['band', 'spatial_ref'], errors='ignore')
        
        # Create DataArray with proper attributes
        da = xr.DataArray(
            data=var_data,
            dims=('time', 'lat', 'lon'),
            attrs=var_dict['attrs']  # Only includes the variable's own name
        )
        ds_out[var_name] = da
    
    # Global attributes
    ds_out.attrs = {
        'description': 'Combined dataset for fire prediction.',
        'created': datetime.date.today().isoformat()
    }
    
    # Save with compression
    encoding = {var: {'zlib': True, 'complevel': 5} for var in ds_out.data_vars}
    ds_out.to_netcdf(output_nc, encoding=encoding)
    print(f"Saved clean NetCDF to {output_nc}")

In [16]:
merge_to_netcdf_with_separate_vars('../geotiffs', '../fire_pred_dataset.nc')

Saved clean NetCDF to ../fire_pred_dataset.nc


In [17]:
from netCDF4 import Dataset
import os

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

In [18]:
data = Dataset("../fire_pred_dataset.nc", mode="r+")

In [19]:
data.dimensions

{'time': "<class 'netCDF4.Dimension'>": name = 'time', size = 62,
 'lon': "<class 'netCDF4.Dimension'>": name = 'lon', size = 446,
 'lat': "<class 'netCDF4.Dimension'>": name = 'lat', size = 210}

In [20]:
fire_mask = data.createVariable('fire_mask', np.int8, ('time', 'lat', 'lon'))

fire_mask.units = 'None'
fire_mask.description = 'Binary mask indicating presence of fire (derived from non-NaN FRP values)'
fire_mask.long_name = 'Fire mask (1=fire, 0=no fire)'

fire_mask[:] = (~np.isnan(data['frp'][:].data)).astype(np.int8)

data.close()